# LAB 3 — Bài 3: Nhận dạng thực thể với BiLSTM-CRF trên PhoNER

> **Phiên bản hướng dẫn dành cho sinh viên mới học**  
> Notebook trình bày đầy đủ quy trình xây dựng hệ thống Named Entity Recognition (NER): đọc dữ liệu theo từng token, kiểm tra nhãn BIO, xây dựng vocabulary, tạo batch có padding, huấn luyện BiLSTM-CRF, đánh giá theo thực thể và phân tích lỗi.

## Mục tiêu học tập

Sau khi hoàn thành bài này, sinh viên có thể:

1. Phân biệt bài toán phân loại câu và bài toán gán nhãn chuỗi.
2. Giải thích vai trò của BiLSTM và CRF trong NER.
3. Tạo `Dataset`, `collate_fn`, mask và độ dài thật cho dữ liệu có độ dài khác nhau.
4. Tránh rò rỉ dữ liệu khi xây dựng vocabulary.
5. Huấn luyện bằng negative log-likelihood của CRF.
6. Đánh giá mô hình theo **thực thể hoàn chỉnh**, không chỉ theo từng token.


## 1. Cài đặt thư viện

Notebook sử dụng:

- `torch`: xây dựng BiLSTM và tối ưu mô hình.
- `pytorch-crf`: triển khai lớp CRF; tên module khi import là `torchcrf`.
- `seqeval`: tính precision, recall, F1 và báo cáo cho bài toán gán nhãn chuỗi.
- `scikit-learn`: tạo confusion matrix ở mức token.
- `matplotlib`: trực quan hóa dữ liệu và quá trình học.

Chỉ cần chạy ô dưới đây khi môi trường chưa có các thư viện tương ứng.


In [ ]:
%pip install -q pytorch-crf seqeval scikit-learn matplotlib


## 2. Import, thiết lập seed và chọn thiết bị

### Vì sao cần random seed?

Việc khởi tạo trọng số, xáo trộn dữ liệu và Dropout đều có yếu tố ngẫu nhiên. Đặt seed giúp kết quả giữa các lần chạy gần giống nhau hơn, thuận tiện khi kiểm tra bài thực hành.

### Lưu ý

Seed giúp tăng tính tái lập nhưng không bảo đảm mọi phần cứng và mọi phiên bản thư viện cho kết quả giống tuyệt đối.


In [ ]:
from collections import Counter
from functools import partial
from pathlib import Path
import json
import random
import time
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from seqeval.metrics import (
    accuracy_score as sequence_accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from seqeval.metrics.sequence_labeling import get_entities
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from torchcrf import CRF

SEED = 42


def set_seed(seed: int = SEED) -> None:
    """Thiết lập seed cho Python, NumPy và PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

print(f"PyTorch version: {torch.__version__}")
print(f"Thiết bị đang sử dụng: {DEVICE}")


## 3. Khai báo và kiểm tra đường dẫn dữ liệu

Notebook giả sử thư mục có cấu trúc:

```text
LAB3_Bai3.ipynb
PhoNER_COVID19/
├── train_word.json
├── dev_word.json
└── test_word.json
```

Mỗi tệp được lưu theo định dạng **JSON Lines**: mỗi dòng là một đối tượng JSON độc lập.

Ta chỉ dùng:

- **Train set**: cập nhật trọng số và xây dựng vocabulary.
- **Dev set**: chọn checkpoint và early stopping.
- **Test set**: đánh giá đúng một lần sau khi hoàn tất lựa chọn mô hình.


In [ ]:
DATA_DIR = Path("PhoNER_COVID19")
TRAIN_PATH = DATA_DIR / "train_word.json"
DEV_PATH = DATA_DIR / "dev_word.json"
TEST_PATH = DATA_DIR / "test_word.json"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "best_bilstm_crf_phoner.pt"

required_paths = [TRAIN_PATH, DEV_PATH, TEST_PATH]
missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    missing_text = "\n".join(f"- {path}" for path in missing_paths)
    raise FileNotFoundError(
        "Không tìm thấy các tệp dữ liệu sau:\n"
        f"{missing_text}\n"
        "Hãy giải nén PhoNER_COVID19 đúng cấu trúc trước khi chạy tiếp."
    )

print("Đã tìm thấy đầy đủ dữ liệu:")
for path in required_paths:
    print(f"- {path}")


## 4. Đọc và kiểm tra dữ liệu JSON Lines

Mỗi mẫu được kỳ vọng có hai trường:

- `words`: danh sách token của câu.
- `tags`: danh sách nhãn tương ứng với từng token.

Hai danh sách phải có cùng độ dài. Không nên bỏ qua bước kiểm tra này vì một mẫu lệch token–nhãn sẽ làm sai toàn bộ bài toán gán nhãn chuỗi.


In [ ]:
def read_jsonlines(path: Path) -> list[dict]:
    """Đọc JSON Lines và kiểm tra cấu trúc từng mẫu."""
    records: list[dict] = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                sample = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"JSON không hợp lệ tại {path}, dòng {line_number}: {error}"
                ) from error

            words = sample.get("words")
            tags = sample.get("tags")

            if not isinstance(words, list) or not isinstance(tags, list):
                raise ValueError(
                    f"Mẫu tại {path}, dòng {line_number} phải có 'words' và 'tags' dạng list."
                )

            if len(words) == 0:
                raise ValueError(f"Mẫu rỗng tại {path}, dòng {line_number}.")

            if len(words) != len(tags):
                raise ValueError(
                    f"Lệch số token và nhãn tại {path}, dòng {line_number}: "
                    f"{len(words)} token nhưng có {len(tags)} nhãn."
                )

            records.append({"words": words, "tags": tags})

    if not records:
        raise ValueError(f"Tệp {path} không chứa mẫu hợp lệ.")

    return records


train_records = read_jsonlines(TRAIN_PATH)
dev_records = read_jsonlines(DEV_PATH)
test_records = read_jsonlines(TEST_PATH)

print(f"Train: {len(train_records):,} câu")
print(f"Dev:   {len(dev_records):,} câu")
print(f"Test:  {len(test_records):,} câu")

print("\nMẫu đầu tiên của train set:")
print("Words:", train_records[0]["words"])
print("Tags: ", train_records[0]["tags"])


## 5. Khảo sát dữ liệu và kiểm tra nhãn BIO

Trong NER, nhãn thường theo quy ước BIO:

- `B-X`: token bắt đầu một thực thể loại `X`.
- `I-X`: token tiếp tục thực thể loại `X`.
- `O`: token không thuộc thực thể nào.

Ví dụ:

```text
Bệnh viện Chợ Rẫy
B-ORGANIZATION I-ORGANIZATION I-ORGANIZATION I-ORGANIZATION
```

Ta khảo sát độ dài câu, số token và phân bố nhãn. Đồng thời kiểm tra sơ bộ trường hợp `I-X` xuất hiện mà không nối tiếp `B-X` hoặc `I-X` cùng loại.


In [ ]:
def summarize_split(name: str, records: list[dict]) -> dict:
    """Tóm tắt số câu và độ dài chuỗi của một tập dữ liệu."""
    lengths = np.array([len(item["words"]) for item in records])
    summary = {
        "split": name,
        "sentences": len(records),
        "tokens": int(lengths.sum()),
        "average_length": float(lengths.mean()),
        "max_length": int(lengths.max()),
    }
    return summary


def find_bio_issues(records: list[dict], maximum_examples: int = 10) -> list[tuple]:
    """Tìm một số vị trí I-X không nối tiếp thực thể X hợp lệ."""
    issues = []

    for sentence_index, sample in enumerate(records):
        previous_prefix = "O"
        previous_type = None

        for token_index, tag in enumerate(sample["tags"]):
            if tag == "O":
                prefix, entity_type = "O", None
            elif "-" in tag:
                prefix, entity_type = tag.split("-", maxsplit=1)
            else:
                prefix, entity_type = tag, None

            if prefix == "I":
                valid_previous = (
                    previous_prefix in {"B", "I"}
                    and previous_type == entity_type
                )
                if not valid_previous:
                    issues.append(
                        (
                            sentence_index,
                            token_index,
                            sample["words"][token_index],
                            tag,
                        )
                    )
                    if len(issues) >= maximum_examples:
                        return issues

            previous_prefix = prefix
            previous_type = entity_type

    return issues


for summary in [
    summarize_split("train", train_records),
    summarize_split("dev", dev_records),
    summarize_split("test", test_records),
]:
    print(
        f"{summary['split']:>5}: "
        f"{summary['sentences']:,} câu | "
        f"{summary['tokens']:,} token | "
        f"dài TB {summary['average_length']:.2f} | "
        f"dài nhất {summary['max_length']}"
    )

bio_issues = find_bio_issues(train_records)
if bio_issues:
    print("\nCảnh báo: tìm thấy một số vị trí BIO cần kiểm tra:")
    for issue in bio_issues:
        print(issue)
else:
    print("\nKhông phát hiện lỗi BIO cơ bản trong train set.")

train_tag_counts = Counter(
    tag for sample in train_records for tag in sample["tags"]
)

plt.figure(figsize=(12, 5))
plt.bar(train_tag_counts.keys(), train_tag_counts.values())
plt.title("Phân bố nhãn token trong train set")
plt.xlabel("Nhãn")
plt.ylabel("Số token")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## 6. Xây dựng vocabulary và bảng nhãn

### Nguyên tắc quan trọng

Vocabulary chỉ được xây từ **train set**. Nếu đọc từ xuất hiện trong dev/test để tạo vocabulary, mô hình đã gián tiếp nhìn thấy thông tin của tập đánh giá; đây là một dạng **data leakage**.

Ta dùng hai token đặc biệt:

- `<pad>`: bù độ dài trong một batch.
- `<unk>`: thay cho token chưa từng xuất hiện trong train set.

### Có nên chuyển toàn bộ về chữ thường?

Tên riêng thường chứa thông tin viết hoa hữu ích. Vì vậy notebook mặc định giữ nguyên hoa–thường. Sinh viên có thể đổi `LOWERCASE = True` để thực nghiệm và so sánh.


In [ ]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
LOWERCASE = False
MIN_FREQUENCY = 1


def normalize_token(token: str, lowercase: bool = LOWERCASE) -> str:
    """Chuẩn hóa Unicode và tùy chọn chuyển token về chữ thường."""
    token = unicodedata.normalize("NFC", str(token).strip())
    return token.lower() if lowercase else token


def build_vocabulary(
    records: list[dict],
    minimum_frequency: int = MIN_FREQUENCY,
) -> tuple[dict[str, int], dict[int, str], Counter]:
    """Xây vocabulary chỉ từ train set."""
    token_counts = Counter(
        normalize_token(token)
        for sample in records
        for token in sample["words"]
    )

    word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}

    sorted_tokens = sorted(
        token_counts.items(),
        key=lambda item: (-item[1], item[0]),
    )

    for token, frequency in sorted_tokens:
        if frequency >= minimum_frequency:
            word2idx[token] = len(word2idx)

    idx2word = {index: token for token, index in word2idx.items()}
    return word2idx, idx2word, token_counts


def build_label_mapping(records: list[dict]) -> tuple[dict[str, int], dict[int, str]]:
    """Xây bảng nhãn từ train set; đưa O về đầu để dễ quan sát."""
    labels = {tag for sample in records for tag in sample["tags"]}
    ordered_labels = (["O"] if "O" in labels else []) + sorted(labels - {"O"})
    label2idx = {label: index for index, label in enumerate(ordered_labels)}
    idx2label = {index: label for label, index in label2idx.items()}
    return label2idx, idx2label


def find_unseen_labels(records: list[dict], label2idx: dict[str, int]) -> set[str]:
    """Tìm nhãn có trong một split nhưng không có trong train set."""
    return {
        tag
        for sample in records
        for tag in sample["tags"]
        if tag not in label2idx
    }


def calculate_oov_rate(records: list[dict], word2idx: dict[str, int]) -> float:
    """Tính tỉ lệ token ngoài vocabulary."""
    tokens = [
        normalize_token(token)
        for sample in records
        for token in sample["words"]
    ]
    number_of_oov = sum(token not in word2idx for token in tokens)
    return number_of_oov / max(len(tokens), 1)


word2idx, idx2word, token_counts = build_vocabulary(train_records)
label2idx, idx2label = build_label_mapping(train_records)

for split_name, records in [("dev", dev_records), ("test", test_records)]:
    unseen_labels = find_unseen_labels(records, label2idx)
    if unseen_labels:
        raise ValueError(
            f"{split_name} set chứa nhãn không xuất hiện trong train set: "
            f"{sorted(unseen_labels)}"
        )

PAD_IDX = word2idx[PAD_TOKEN]
UNK_IDX = word2idx[UNK_TOKEN]
LABEL_PAD_IDX = label2idx.get("O", 0)  # Chỉ dùng tại vị trí mask=False.

print(f"Kích thước vocabulary: {len(word2idx):,}")
print(f"Số nhãn: {len(label2idx)}")
print("Các nhãn:", list(label2idx.keys()))
print(f"OOV trên dev:  {calculate_oov_rate(dev_records, word2idx):.2%}")
print(f"OOV trên test: {calculate_oov_rate(test_records, word2idx):.2%}")


## 7. Kiểm tra kết quả mã hóa một câu

Trước khi xây mô hình, cần kiểm tra thủ công:

- Token gốc.
- Token sau chuẩn hóa.
- Chỉ số token.
- Chỉ số nhãn.
- Kết quả giải mã ngược.

Bước này giúp phát hiện sớm lỗi vocabulary, lỗi `<unk>` hoặc lỗi căn chỉnh token–nhãn.


In [ ]:
def encode_tokens(tokens: list[str], word2idx: dict[str, int]) -> list[int]:
    """Chuyển danh sách token thành danh sách chỉ số."""
    return [
        word2idx.get(normalize_token(token), UNK_IDX)
        for token in tokens
    ]


def encode_tags(tags: list[str], label2idx: dict[str, int]) -> list[int]:
    """Chuyển danh sách nhãn thành danh sách chỉ số."""
    return [label2idx[tag] for tag in tags]


example = train_records[0]
example_token_ids = encode_tokens(example["words"], word2idx)
example_tag_ids = encode_tags(example["tags"], label2idx)

print("Token gốc:       ", example["words"])
print("Token chuẩn hóa: ", [normalize_token(token) for token in example["words"]])
print("Token IDs:       ", example_token_ids)
print("Tag IDs:         ", example_tag_ids)
print("Giải mã token:   ", [idx2word[index] for index in example_token_ids])
print("Giải mã nhãn:    ", [idx2label[index] for index in example_tag_ids])


## 8. Xây dựng `Dataset`

Một mẫu NER phải giữ đồng thời:

- Chuỗi token đã mã hóa.
- Chuỗi nhãn có cùng độ dài.
- Token gốc để phục vụ phân tích lỗi.

`Dataset` chưa padding từng câu. Padding chỉ nên được thực hiện trong `collate_fn` theo câu dài nhất của **từng batch**, giúp tiết kiệm bộ nhớ hơn so với padding toàn bộ dữ liệu về độ dài lớn nhất của tập.


In [ ]:
class PhoNERDataset(Dataset):
    """Dataset cho dữ liệu NER theo từng token."""

    def __init__(
        self,
        records: list[dict],
        word2idx: dict[str, int],
        label2idx: dict[str, int],
    ) -> None:
        self.samples = []

        for sample in records:
            tokens = sample["words"]
            tags = sample["tags"]

            token_ids = encode_tokens(tokens, word2idx)
            tag_ids = encode_tags(tags, label2idx)

            self.samples.append(
                {
                    "tokens": tokens,
                    "input_ids": torch.tensor(token_ids, dtype=torch.long),
                    "labels": torch.tensor(tag_ids, dtype=torch.long),
                }
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> dict:
        return self.samples[index]


train_dataset = PhoNERDataset(train_records, word2idx, label2idx)
dev_dataset = PhoNERDataset(dev_records, word2idx, label2idx)
test_dataset = PhoNERDataset(test_records, word2idx, label2idx)

sample = train_dataset[0]
print("Số mẫu train:", len(train_dataset))
print("input_ids shape:", sample["input_ids"].shape)
print("labels shape:   ", sample["labels"].shape)
print("Số token gốc:  ", len(sample["tokens"]))


## 9. Tạo batch có padding bằng `collate_fn`

Các câu có độ dài khác nhau nên không thể ghép trực tiếp thành một tensor hai chiều. `collate_fn` tạo:

- `input_ids`: token đã padding.
- `labels`: nhãn đã padding; giá trị padding này không được tính vì có mask.
- `lengths`: độ dài thật của từng câu.
- `mask`: `True` tại token thật và `False` tại padding.

`lengths` được dùng cho packed sequence của BiLSTM. `mask` được dùng cho CRF và khi khôi phục nhãn thật.


In [ ]:
def collate_ner_batch(
    batch: list[dict],
    pad_idx: int,
    label_pad_idx: int,
) -> dict:
    """Padding các chuỗi trong batch theo độ dài câu dài nhất."""
    input_sequences = [item["input_ids"] for item in batch]
    label_sequences = [item["labels"] for item in batch]
    lengths = torch.tensor(
        [len(sequence) for sequence in input_sequences],
        dtype=torch.long,
    )

    input_ids = pad_sequence(
        input_sequences,
        batch_first=True,
        padding_value=pad_idx,
    )
    labels = pad_sequence(
        label_sequences,
        batch_first=True,
        padding_value=label_pad_idx,
    )

    positions = torch.arange(input_ids.size(1)).unsqueeze(0)
    mask = positions < lengths.unsqueeze(1)

    return {
        "input_ids": input_ids,
        "labels": labels,
        "lengths": lengths,
        "mask": mask,
        "tokens": [item["tokens"] for item in batch],
    }


collate_fn = partial(
    collate_ner_batch,
    pad_idx=PAD_IDX,
    label_pad_idx=LABEL_PAD_IDX,
)

example_batch = collate_fn([train_dataset[0], train_dataset[1]])
print("input_ids:", example_batch["input_ids"].shape)
print("labels:   ", example_batch["labels"].shape)
print("lengths: ", example_batch["lengths"])
print("mask:     ", example_batch["mask"].shape)
print("Số token thật:", example_batch["mask"].sum(dim=1))


## 10. Tạo `DataLoader`

- Train loader dùng `shuffle=True` để thay đổi thứ tự mẫu ở mỗi epoch.
- Dev/test loader giữ nguyên thứ tự để thuận tiện phân tích.
- `pin_memory=True` chỉ hữu ích khi huấn luyện trên GPU.
- `num_workers=0` tương thích tốt với notebook trên Windows và Colab; có thể tăng khi môi trường hỗ trợ.


In [ ]:
BATCH_SIZE = 16

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=PIN_MEMORY,
    num_workers=0,
    generator=loader_generator,
)
dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=PIN_MEMORY,
    num_workers=0,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=PIN_MEMORY,
    num_workers=0,
)

first_batch = next(iter(train_loader))
print("Kích thước batch đầu tiên:")
print("input_ids:", tuple(first_batch["input_ids"].shape))
print("labels:   ", tuple(first_batch["labels"].shape))
print("mask:     ", tuple(first_batch["mask"].shape))
print("lengths:  ", first_batch["lengths"].tolist())


## 11. Lý thuyết mô hình BiLSTM-CRF

### 11.1. Embedding

Mỗi token ID được ánh xạ thành một vector liên tục. Các vector này được học cùng mô hình.

### 11.2. Bidirectional LSTM

BiLSTM đọc chuỗi theo hai hướng:

- Hướng thuận sử dụng ngữ cảnh bên trái.
- Hướng ngược sử dụng ngữ cảnh bên phải.

Với NER, cả hai phía đều quan trọng. Chẳng hạn từ “Rẫy” dễ được nhận diện là một phần tên tổ chức khi nhìn thấy các token “Bệnh viện Chợ” ở bên trái.

### 11.3. Emission score

Tại mỗi vị trí, tầng tuyến tính tạo điểm số cho mọi nhãn. Điểm này cho biết mức phù hợp giữa token hiện tại và từng nhãn.

### 11.4. Conditional Random Field

Nếu dự đoán từng token độc lập, mô hình có thể tạo chuỗi không hợp lý như `O → I-ORGANIZATION`. CRF học thêm điểm chuyển giữa các nhãn và tìm **toàn bộ chuỗi nhãn tốt nhất** bằng giải mã Viterbi.

### 11.5. Vì sao cần packed sequence?

Nếu chỉ padding rồi chạy BiLSTM bình thường, nhánh LSTM chạy ngược có thể đọc các token padding trước khi tới token thật. Packed sequence giúp mạng chỉ xử lý độ dài thật của mỗi câu.


In [ ]:
class BiLSTMCRF(nn.Module):
    """Mô hình Embedding → BiLSTM → Linear → CRF."""

    def __init__(
        self,
        vocab_size: int,
        number_of_labels: int,
        pad_idx: int,
        embedding_dim: int = 256,
        hidden_dim: int = 256,
        number_of_layers: int = 2,
        dropout: float = 0.30,
    ) -> None:
        super().__init__()

        if number_of_layers < 1:
            raise ValueError("number_of_layers phải lớn hơn hoặc bằng 1.")

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx,
        )
        self.input_dropout = nn.Dropout(dropout)

        self.bilstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=number_of_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if number_of_layers > 1 else 0.0,
        )

        self.output_dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, number_of_labels)
        self.crf = CRF(number_of_labels, batch_first=True)

    def calculate_emissions(
        self,
        input_ids: torch.Tensor,
        lengths: torch.Tensor,
    ) -> torch.Tensor:
        """Tạo emission score cho mỗi token và mỗi nhãn."""
        embeddings = self.input_dropout(self.embedding(input_ids))

        packed_embeddings = pack_padded_sequence(
            embeddings,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_outputs, _ = self.bilstm(packed_embeddings)
        sequence_outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=input_ids.size(1),
        )

        sequence_outputs = self.output_dropout(sequence_outputs)
        return self.classifier(sequence_outputs)

    def negative_log_likelihood(
        self,
        input_ids: torch.Tensor,
        lengths: torch.Tensor,
        tags: torch.Tensor,
        mask: torch.Tensor,
    ) -> torch.Tensor:
        """Tính negative log-likelihood trung bình của CRF."""
        emissions = self.calculate_emissions(input_ids, lengths)
        log_likelihood = self.crf(
            emissions,
            tags,
            mask=mask.bool(),
            reduction="mean",
        )
        return -log_likelihood

    def decode(
        self,
        input_ids: torch.Tensor,
        lengths: torch.Tensor,
        mask: torch.Tensor,
    ) -> list[list[int]]:
        """Giải mã chuỗi nhãn tốt nhất bằng Viterbi."""
        emissions = self.calculate_emissions(input_ids, lengths)
        return self.crf.decode(emissions, mask=mask.bool())


## 12. Chọn mô hình và siêu tham số

Bản sinh viên dùng 5 tầng BiLSTM với hidden size lớn. Cấu hình đó tốn bộ nhớ, huấn luyện chậm và có nguy cơ overfitting khi chưa có thực nghiệm chứng minh.

Notebook bắt đầu bằng cấu hình vừa phải:

- 2 tầng BiLSTM.
- Embedding 256 chiều.
- Hidden state 256 chiều cho mỗi hướng.
- Dropout 0.30.

Đây không phải cấu hình duy nhất đúng. Sinh viên cần thay đổi từng siêu tham số và so sánh trên dev set.


In [ ]:
MODEL_CONFIG = {
    "embedding_dim": 256,
    "hidden_dim": 256,
    "number_of_layers": 2,
    "dropout": 0.30,
}

model = BiLSTMCRF(
    vocab_size=len(word2idx),
    number_of_labels=len(label2idx),
    pad_idx=PAD_IDX,
    **MODEL_CONFIG,
).to(DEVICE)

number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(model)
print(f"\nSố tham số có thể học: {number_of_parameters:,}")

# Kiểm tra một lượt forward trước khi huấn luyện.
sanity_batch = next(iter(train_loader))
sanity_input_ids = sanity_batch["input_ids"].to(DEVICE)
sanity_labels = sanity_batch["labels"].to(DEVICE)
sanity_lengths = sanity_batch["lengths"]
sanity_mask = sanity_batch["mask"].to(DEVICE)

with torch.no_grad():
    sanity_loss = model.negative_log_likelihood(
        sanity_input_ids,
        sanity_lengths,
        sanity_labels,
        sanity_mask,
    )
    sanity_predictions = model.decode(
        sanity_input_ids,
        sanity_lengths,
        sanity_mask,
    )

print(f"Sanity loss: {sanity_loss.item():.4f}")
print("Độ dài dự đoán câu đầu:", len(sanity_predictions[0]))
print("Độ dài thật câu đầu:    ", sanity_lengths[0].item())


## 13. Hàm mất mát, optimizer và gradient clipping

### Hàm mất mát

CRF tối đa hóa xác suất của chuỗi nhãn đúng. Vì optimizer của PyTorch thực hiện cực tiểu hóa, ta dùng **negative log-likelihood**.

### Optimizer

`AdamW` là lựa chọn khởi đầu phổ biến. `weight_decay` tạo regularization nhẹ.

### Gradient clipping

RNN có thể gặp exploding gradient. Ta giới hạn chuẩn gradient trước khi cập nhật trọng số.


In [ ]:
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_GRADIENT_NORM = 5.0

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print(optimizer)


## 14. Hàm huấn luyện một epoch

Trong mỗi batch:

1. Chuyển tensor sang thiết bị.
2. Xóa gradient cũ.
3. Tính negative log-likelihood.
4. Lan truyền ngược.
5. Clip gradient.
6. Cập nhật trọng số.

Loss được cộng theo số câu để tính trung bình chính xác cho toàn bộ epoch.


In [ ]:
def train_one_epoch(
    model: BiLSTMCRF,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    maximum_gradient_norm: float,
) -> float:
    """Huấn luyện mô hình trong một epoch và trả về loss trung bình."""
    model.train()
    total_loss = 0.0
    total_sequences = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        lengths = batch["lengths"]
        mask = batch["mask"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        loss = model.negative_log_likelihood(
            input_ids,
            lengths,
            labels,
            mask,
        )

        if not torch.isfinite(loss):
            raise RuntimeError(f"Loss không hữu hạn: {loss.item()}")

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), maximum_gradient_norm)
        optimizer.step()

        batch_size = input_ids.size(0)
        total_loss += loss.item() * batch_size
        total_sequences += batch_size

    return total_loss / max(total_sequences, 1)


## 15. Hàm đánh giá

NER nên được đánh giá theo **thực thể hoàn chỉnh**. Một thực thể chỉ được tính đúng khi cả loại thực thể và ranh giới bắt đầu–kết thúc đều đúng.

Các chỉ số chính:

- Precision: trong các thực thể mô hình dự đoán, bao nhiêu thực thể đúng.
- Recall: trong các thực thể thật, mô hình tìm được bao nhiêu.
- F1: trung bình điều hòa của precision và recall.

`seqeval` nhận đầu vào là danh sách các chuỗi nhãn dạng văn bản, vì vậy ta phải loại padding rồi chuyển ID về nhãn.


In [ ]:
def evaluate_model(
    model: BiLSTMCRF,
    loader: DataLoader,
    device: torch.device,
) -> tuple[dict, list[list[str]], list[list[str]], list[list[str]]]:
    """Tính loss và metric theo thực thể cho một DataLoader."""
    model.eval()

    total_loss = 0.0
    total_sequences = 0
    all_predictions: list[list[str]] = []
    all_gold_labels: list[list[str]] = []
    all_tokens: list[list[str]] = []

    with torch.inference_mode():
        for batch in loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            lengths = batch["lengths"]
            mask = batch["mask"].to(device, non_blocking=True)

            loss = model.negative_log_likelihood(
                input_ids,
                lengths,
                labels,
                mask,
            )
            predicted_ids = model.decode(input_ids, lengths, mask)

            batch_size = input_ids.size(0)
            total_loss += loss.item() * batch_size
            total_sequences += batch_size

            labels_cpu = batch["labels"]
            mask_cpu = batch["mask"]

            for prediction, gold, valid_mask, tokens in zip(
                predicted_ids,
                labels_cpu,
                mask_cpu,
                batch["tokens"],
            ):
                real_length = int(valid_mask.sum().item())
                prediction = prediction[:real_length]
                gold = gold[:real_length].tolist()

                if len(prediction) != real_length:
                    raise RuntimeError(
                        "Độ dài chuỗi CRF decode không khớp với mask."
                    )

                all_predictions.append([idx2label[index] for index in prediction])
                all_gold_labels.append([idx2label[index] for index in gold])
                all_tokens.append(tokens[:real_length])

    metrics = {
        "loss": total_loss / max(total_sequences, 1),
        "accuracy": sequence_accuracy_score(all_gold_labels, all_predictions),
        "precision": precision_score(all_gold_labels, all_predictions),
        "recall": recall_score(all_gold_labels, all_predictions),
        "f1": f1_score(all_gold_labels, all_predictions),
    }

    return metrics, all_predictions, all_gold_labels, all_tokens


## 16. Huấn luyện và lưu checkpoint tốt nhất

Sau mỗi epoch, mô hình được đánh giá trên dev set. Ta lưu checkpoint khi **dev F1** tăng.

Early stopping dừng quá trình khi F1 không cải thiện trong một số epoch liên tiếp. Việc này giúp giảm thời gian và hạn chế overfitting.

Checkpoint lưu cả:

- Trọng số mô hình.
- Trạng thái optimizer.
- Vocabulary và bảng nhãn.
- Cấu hình mô hình.
- Epoch và dev F1 tốt nhất.


In [ ]:
NUMBER_OF_EPOCHS = 20
PATIENCE = 4
MINIMUM_IMPROVEMENT = 1e-4

history = {
    "train_loss": [],
    "dev_loss": [],
    "dev_f1": [],
}

best_dev_f1 = -1.0
epochs_without_improvement = 0
training_start_time = time.perf_counter()

for epoch in range(1, NUMBER_OF_EPOCHS + 1):
    epoch_start_time = time.perf_counter()

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        DEVICE,
        MAX_GRADIENT_NORM,
    )
    dev_metrics, _, _, _ = evaluate_model(model, dev_loader, DEVICE)

    history["train_loss"].append(train_loss)
    history["dev_loss"].append(dev_metrics["loss"])
    history["dev_f1"].append(dev_metrics["f1"])

    epoch_seconds = time.perf_counter() - epoch_start_time
    print(
        f"Epoch {epoch:02d}/{NUMBER_OF_EPOCHS} | "
        f"train loss {train_loss:.4f} | "
        f"dev loss {dev_metrics['loss']:.4f} | "
        f"dev P/R/F1 "
        f"{dev_metrics['precision']:.4f}/"
        f"{dev_metrics['recall']:.4f}/"
        f"{dev_metrics['f1']:.4f} | "
        f"{epoch_seconds:.1f}s"
    )

    improved = dev_metrics["f1"] > best_dev_f1 + MINIMUM_IMPROVEMENT

    if improved:
        best_dev_f1 = dev_metrics["f1"]
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "best_dev_f1": best_dev_f1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "model_config": MODEL_CONFIG,
                "word2idx": word2idx,
                "label2idx": label2idx,
                "lowercase": LOWERCASE,
            },
            CHECKPOINT_PATH,
        )
        print(f"  → Đã lưu checkpoint tốt nhất tại {CHECKPOINT_PATH}")
    else:
        epochs_without_improvement += 1
        print(
            f"  → Chưa cải thiện: "
            f"{epochs_without_improvement}/{PATIENCE} epoch"
        )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping: dừng huấn luyện.")
        break

training_seconds = time.perf_counter() - training_start_time
print(f"\nDev F1 tốt nhất: {best_dev_f1:.4f}")
print(f"Tổng thời gian huấn luyện: {training_seconds / 60:.2f} phút")


## 17. Vẽ biểu đồ quá trình học

Ta quan sát riêng:

- Train loss và dev loss: giúp phát hiện underfitting hoặc overfitting.
- Dev F1: chỉ số chính để chọn checkpoint.

Ví dụ, train loss tiếp tục giảm nhưng dev loss tăng và dev F1 giảm thường là dấu hiệu overfitting.


In [ ]:
epoch_numbers = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epoch_numbers, history["train_loss"], marker="o", label="Train loss")
plt.plot(epoch_numbers, history["dev_loss"], marker="o", label="Dev loss")
plt.title("Loss theo epoch")
plt.xlabel("Epoch")
plt.ylabel("Negative log-likelihood")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epoch_numbers, history["dev_f1"], marker="o")
plt.title("Entity-level F1 trên dev set")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.ylim(0.0, 1.0)
plt.grid(True)
plt.tight_layout()
plt.show()


## 18. Tải lại checkpoint tốt nhất

Không đánh giá test bằng trọng số của epoch cuối cùng vì epoch cuối chưa chắc tốt nhất. Ta khởi tạo lại mô hình từ cấu hình đã lưu và nạp checkpoint có dev F1 cao nhất.

Hàm tải dưới đây tương thích với cả các phiên bản PyTorch có và chưa có tham số `weights_only`.


In [ ]:
def load_checkpoint(path: Path, device: torch.device) -> dict:
    """Tải checkpoint với khả năng tương thích nhiều phiên bản PyTorch."""
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Chưa có checkpoint tại {CHECKPOINT_PATH}. Hãy chạy bước huấn luyện trước."
    )

checkpoint = load_checkpoint(CHECKPOINT_PATH, DEVICE)

if checkpoint["word2idx"] != word2idx or checkpoint["label2idx"] != label2idx:
    raise ValueError(
        "Vocabulary hoặc bảng nhãn hiện tại không khớp checkpoint. "
        "Hãy chạy notebook theo đúng thứ tự từ đầu."
    )

best_model = BiLSTMCRF(
    vocab_size=len(word2idx),
    number_of_labels=len(label2idx),
    pad_idx=PAD_IDX,
    **checkpoint["model_config"],
).to(DEVICE)
best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()

print(f"Đã tải checkpoint ở epoch {checkpoint['epoch']}")
print(f"Dev F1 của checkpoint: {checkpoint['best_dev_f1']:.4f}")


## 19. Đánh giá cuối cùng trên test set

Test set chỉ được sử dụng sau khi đã chọn mô hình bằng dev set.

Báo cáo `seqeval` nhóm kết quả theo **loại thực thể** thay vì từng nhãn `B-` và `I-`. Đây là cách đánh giá phù hợp hơn cho NER.


In [ ]:
test_metrics, test_predictions, test_gold_labels, test_tokens = evaluate_model(
    best_model,
    test_loader,
    DEVICE,
)

print("=== KẾT QUẢ TRÊN TEST SET ===")
print(f"Loss:      {test_metrics['loss']:.4f}")
print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1:        {test_metrics['f1']:.4f}")

print("\n=== BÁO CÁO THEO LOẠI THỰC THỂ ===")
print(
    classification_report(
        test_gold_labels,
        test_predictions,
        digits=4,
        zero_division=0,
    )
)


## 20. Confusion matrix ở mức token

Entity-level F1 là chỉ số chính, nhưng confusion matrix theo token vẫn giúp trả lời các câu hỏi như:

- Mô hình có thường nhầm `B-X` thành `I-X` không?
- Một loại thực thể có thường bị dự đoán thành `O` không?
- Hai loại thực thể nào dễ bị nhầm với nhau?

Lưu ý: confusion matrix này đánh giá từng token độc lập, vì vậy không thay thế entity-level F1.


In [ ]:
flat_gold = [tag for sequence in test_gold_labels for tag in sequence]
flat_predictions = [tag for sequence in test_predictions for tag in sequence]
label_names = [idx2label[index] for index in range(len(idx2label))]

matrix = confusion_matrix(
    flat_gold,
    flat_predictions,
    labels=label_names,
)

figure_size = max(10, min(18, len(label_names) * 0.65))
plt.figure(figsize=(figure_size, figure_size))
display = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=label_names,
)
display.plot(
    ax=plt.gca(),
    xticks_rotation=90,
    values_format="d",
)
plt.title("Confusion matrix theo token trên test set")
plt.tight_layout()
plt.show()


## 21. Xem một số câu dự đoán sai

Chỉ nhìn một con số F1 chưa đủ để cải thiện mô hình. Phân tích lỗi giúp phát hiện:

- Sai ranh giới thực thể.
- Đúng ranh giới nhưng sai loại.
- Bỏ sót thực thể.
- Dự đoán thực thể không tồn tại.
- Lỗi do token ngoài vocabulary.

`get_entities` trả về bộ `(loại thực thể, vị trí bắt đầu, vị trí kết thúc)`.


In [ ]:
def show_prediction_errors(
    tokens_list: list[list[str]],
    gold_sequences: list[list[str]],
    predicted_sequences: list[list[str]],
    maximum_examples: int = 5,
) -> None:
    """Hiển thị một số câu có chuỗi nhãn dự đoán khác nhãn thật."""
    shown = 0

    for tokens, gold, predicted in zip(
        tokens_list,
        gold_sequences,
        predicted_sequences,
    ):
        if gold == predicted:
            continue

        print("=" * 100)
        print("Câu:", " ".join(tokens))
        print("Thực thể thật:    ", get_entities(gold))
        print("Thực thể dự đoán: ", get_entities(predicted))
        print("\nCác token bị gán sai:")

        for index, (token, gold_tag, predicted_tag) in enumerate(
            zip(tokens, gold, predicted)
        ):
            if gold_tag != predicted_tag:
                normalized = normalize_token(token)
                oov_marker = " [OOV]" if normalized not in word2idx else ""
                print(
                    f"  {index:>3} | {token:<25} | "
                    f"gold={gold_tag:<25} | pred={predicted_tag:<25}"
                    f"{oov_marker}"
                )

        shown += 1
        if shown >= maximum_examples:
            break

    if shown == 0:
        print("Không tìm thấy câu dự đoán sai trong phạm vi dữ liệu.")


show_prediction_errors(
    test_tokens,
    test_gold_labels,
    test_predictions,
    maximum_examples=5,
)


## 22. Dự đoán một câu mới

Mô hình được huấn luyện ở mức token, vì vậy hàm dự đoán nhận vào **danh sách token**. Không nên mặc định `sentence.split()` là bộ tách từ hoàn hảo cho tiếng Việt.

Trong ứng dụng thực tế, cần dùng cùng quy trình tokenization như dữ liệu huấn luyện. Ví dụ dưới đây dùng token của một câu trong test set để bảo đảm đúng định dạng.


In [ ]:
def predict_tokens(
    model: BiLSTMCRF,
    tokens: list[str],
) -> tuple[list[tuple[str, str]], list[tuple]]:
    """Dự đoán nhãn cho một danh sách token đã được tách sẵn."""
    if not tokens:
        raise ValueError("Danh sách token không được rỗng.")

    token_ids = torch.tensor(
        [encode_tokens(tokens, word2idx)],
        dtype=torch.long,
        device=DEVICE,
    )
    lengths = torch.tensor([len(tokens)], dtype=torch.long)
    mask = torch.ones_like(token_ids, dtype=torch.bool, device=DEVICE)

    model.eval()
    with torch.inference_mode():
        predicted_ids = model.decode(token_ids, lengths, mask)[0]

    predicted_tags = [idx2label[index] for index in predicted_ids]
    token_tag_pairs = list(zip(tokens, predicted_tags))
    entities = get_entities(predicted_tags)
    return token_tag_pairs, entities


example_tokens = test_records[0]["words"]
pairs, entities = predict_tokens(best_model, example_tokens)

print("Token và nhãn dự đoán:")
for token, tag in pairs:
    print(f"{token:<30} {tag}")

print("\nCác thực thể dự đoán:", entities)

# Có thể thử với token tự nhập, nhưng cần tách token giống PhoNER:
# custom_tokens = ["Bệnh", "nhân", "được", "điều", "trị", "tại", "Bệnh", "viện", "Chợ", "Rẫy", "."]
# print(predict_tokens(best_model, custom_tokens))


## 23. Tổng kết kiến thức

Trong bài này, chúng ta đã thực hiện toàn bộ pipeline NER:

1. Đọc JSON Lines và kiểm tra token–nhãn.
2. Khảo sát độ dài, phân bố nhãn và tính hợp lệ cơ bản của BIO.
3. Xây vocabulary **chỉ từ train set** để tránh data leakage.
4. Dùng `<pad>`, `<unk>`, `lengths` và `mask` đúng mục đích.
5. Dùng packed sequence để BiLSTM không học từ padding.
6. Dùng CRF để mô hình hóa quan hệ chuyển tiếp giữa các nhãn.
7. Chọn checkpoint bằng dev entity-level F1 và early stopping.
8. Đánh giá test bằng `seqeval`, confusion matrix và phân tích lỗi.

### Các lỗi thường gặp

- Xây vocabulary từ cả train, dev và test.
- Padding toàn bộ dữ liệu về độ dài lớn nhất của cả tập.
- Không truyền mask cho CRF.
- Dùng giá trị padding như một nhãn thật trong metric.
- Đánh giá NER chỉ bằng accuracy từng token; nhãn `O` nhiều có thể làm accuracy cao giả tạo.
- Đánh giá test ở mọi epoch rồi chọn mô hình theo test.
- Dùng mô hình quá sâu mà chưa kiểm tra overfitting và chi phí huấn luyện.

### Bài tập mở rộng

1. So sánh giữ nguyên chữ hoa với `LOWERCASE = True`.
2. Thử `hidden_dim` bằng 128, 256 và 384.
3. So sánh 1, 2 và 3 tầng BiLSTM.
4. Thêm pretrained word embedding hoặc mô hình ngôn ngữ tiếng Việt.
5. So sánh BiLSTM có CRF với BiLSTM dùng CrossEntropy theo từng token.
6. Thống kê F1 riêng cho thực thể dài một token và thực thể nhiều token.
7. Phân tích ảnh hưởng của OOV đến recall của từng loại thực thể.
